# 02 · Churn Prediction Model

**Data:** `telco_customers.parquet` (labeled `churn_flag`)  
**Models:** XGBoost, LightGBM, MLP (neural baseline)  
**Output:** P(churn) — proxy for 30-day churn risk

Split: 60% train · 20% validation · 20% test (stratified).


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "scripts" / "build_datasets.py").exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data" / "modeling"
MODELS = PROJECT_ROOT / "models"
MODELS.mkdir(exist_ok=True)


def load_parquet(name: str) -> pd.DataFrame:
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path} — run: python scripts/uci_pipeline.py")
    return pd.read_parquet(path)


def save_artifact(name: str, obj) -> Path:
    path = MODELS / name
    joblib.dump(obj, path)
    print(f"Saved → {path.relative_to(PROJECT_ROOT)}")
    return path


def audit_and_clean(
    df: pd.DataFrame,
    *,
    subset: list[str] | None = None,
    id_col: str | None = None,
    required_cols: list[str] | None = None,
    label: str = "dataset",
) -> pd.DataFrame:
    """Report and drop duplicate rows + rows with NA in required columns (before split)."""
    out = df.copy()
    n0 = len(out)
    dup_subset = subset if subset is not None else ([id_col] if id_col else None)
    n_dup = out.duplicated(subset=dup_subset, keep="first").sum() if dup_subset else out.duplicated(keep="first").sum()
    if n_dup:
        out = out.drop_duplicates(subset=dup_subset, keep="first")
    req = [c for c in (required_cols or []) if c in out.columns]
    na_rows = out[req].isna().any(axis=1).sum() if req else 0
    na_by_col = out[req].isna().sum()
    if req:
        out = out.dropna(subset=req)
    print(
        f"[{label}] {n0:,} rows -> {len(out):,} | "
        f"dropped {n_dup:,} duplicates, {na_rows:,} rows with NA"
    )
    if na_rows and (na_by_col > 0).any():
        print("  NA counts:", na_by_col[na_by_col > 0].to_dict())
    return out


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    RocCurveDisplay, accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import lightgbm as lgb
import xgboost as xgb

df = load_parquet("telco_customers.parquet")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["SeniorCitizen"] = df["SeniorCitizen"].astype(int)

TARGET = "churn_flag"
DROP = ["telco_customer_id", "Churn", TARGET]
feature_cols = [c for c in df.columns if c not in DROP]
df = audit_and_clean(
    df,
    id_col="telco_customer_id",
    required_cols=["telco_customer_id", TARGET, *feature_cols],
    label="telco_customers",
)
print(f"Rows: {len(df):,} · Churn rate: {df['churn_flag'].mean():.1%}")

X = df.drop(columns=[c for c in DROP if c in df.columns])
y = df[TARGET]


In [ ]:
# Correlation (numeric only)
num = X.select_dtypes(include=[np.number])
num_corr = num.assign(churn_flag=y).corr()["churn_flag"].drop("churn_flag").sort_values(key=abs, ascending=False)
fig, ax = plt.subplots(figsize=(7, 5))
num_corr.head(15).plot(kind="barh", ax=ax, color="#1f4e79")
ax.set_title("Top numeric correlations with churn")
plt.tight_layout()
plt.show()
display(num_corr.head(10).to_frame("corr_with_churn"))


In [ ]:
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat_cols),
])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)
print(f"Train {len(X_train):,} · Val {len(X_val):,} · Test {len(X_test):,}")


In [ ]:
def eval_classifier(name, model, X_tr, y_tr, X_v, y_v, X_te, y_te):
    model.fit(X_tr, y_tr)
    prob_v = model.predict_proba(X_v)[:, 1]
    prob_t = model.predict_proba(X_te)[:, 1]
    pred_t = (prob_t >= 0.5).astype(int)
    return {
        "model": name,
        "val_roc_auc": roc_auc_score(y_v, prob_v),
        "test_roc_auc": roc_auc_score(y_te, prob_t),
        "test_f1": f1_score(y_te, pred_t),
        "test_precision": precision_score(y_te, pred_t),
        "test_recall": recall_score(y_te, pred_t),
        "estimator": model,
    }

models = {
    "XGBoost": xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
        random_state=RANDOM_STATE, verbosity=0,
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
    ),
    "MLP": Pipeline([
        ("prep", preprocess),
        ("clf", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=RANDOM_STATE)),
    ]),
}

results = []
for name, est in models.items():
    if name == "MLP":
        results.append(eval_classifier(name, est, X_train, y_train, X_val, y_val, X_test, y_test))
    else:
        pipe = Pipeline([("prep", preprocess), ("clf", est)])
        results.append(eval_classifier(name, pipe, X_train, y_train, X_val, y_val, X_test, y_test))

leaderboard = pd.DataFrame([{k: v for k, v in r.items() if k != "estimator"} for r in results]).set_index("model")
display(leaderboard.round(4))
best = max(results, key=lambda r: r["val_roc_auc"])
print(f"Best model: {best['model']} (val ROC-AUC={best['val_roc_auc']:.4f})")


In [ ]:
# Feature importance (tree model)
best_pipe = best["estimator"]
if best["model"] in ("XGBoost", "LightGBM"):
    prep = best_pipe.named_steps["prep"]
    clf = best_pipe.named_steps["clf"]
    X_train_t = prep.fit_transform(X_train, y_train)
    feat_names = prep.get_feature_names_out()
    imp = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(8, 5))
    imp.iloc[::-1].plot(kind="barh", ax=ax, color="#c0392b")
    ax.set_title(f"Top 20 feature importances — {best['model']}")
    plt.tight_layout()
    plt.show()

RocCurveDisplay.from_predictions(y_test, best_pipe.predict_proba(X_test)[:, 1], name=best["model"])
plt.title("ROC — hold-out test set")
plt.show()

save_artifact("02_churn_best.joblib", {
    "model_name": best["model"],
    "pipeline": best_pipe,
    "metrics": {k: best[k] for k in leaderboard.columns},
    "target": "churn_flag",
})
